# B3：SUMO CACC 编队协同巡航 —— 真 TraCI 闭环
# B3: SUMO CACC Platooning — Real TraCI Closed Loop

## 实验目标 | Objectives
- 用 TraCI 实时控制 SUMO 中的车辆
  Control SUMO vehicles in real time via TraCI
- 实现 CACC 编队跟驰算法，验证**弦稳定性**
  Implement CACC platoon control and verify string stability

## 背景知识 | Background
**CACC（协同自适应巡航）**：编队中每辆车跟随前车，保持固定时距。
**弦稳定性（String Stability）**：头车的扰动（如急刹）不应在向后传递时被放大，
否则编队会" accordion"（手风琴效应）甚至追尾。
本实验用 **SUMO 真实交通仿真器** + TraCI 接口做真闭环。

**TraCI** = Traffic Control Interface，SUMO 的 Python 实时控制接口，
每步可读取车辆状态、设置速度/变道。

## 通过判据 | Pass Criteria
```python
assert max_spacing_error <= 2.0  # 编队稳定，间距误差 ≤ 2m
```


In [ ]:
import os, sys
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
print('✓ 导入成功')


## 1. CACC 控制器 | CACC Controller
固定时距策略：期望间距 = 自车速度 × headway + 5m。
PD 控制同时修正间距误差和速度误差。


In [ ]:
class CACCController(BaseController):
    def __init__(self, config=None):
        super().__init__(config)
        self.headway = self.config.get('headway', 0.8)
        self.kp, self.kd = 1.2, 0.8
    def step(self, observation):
        ego, targets = observation['ego'], observation.get('targets', [])
        if not targets:
            return {'target_a': 0.0, 'steer_angle': 0.0}
        lead = targets[0]
        gap_err = (lead['x'] - ego['x'] - 4.5) - ego['v'] * self.headway
        v_err = lead['v'] - ego['v']
        a = self.kp * gap_err + self.kd * v_err
        return {'target_a': max(-3.0, min(2.0, a)), 'steer_angle': 0.0}
print('✓ CACC 控制器就绪')


## 2. 生成 SUMO 路网 | Generate SUMO Network
用 `netgenerate` 生成直道路网，4 辆车组成编队（头车 + 3 跟随车）。


In [ ]:
import os
workdir = '/tmp/b3_sumo'
os.makedirs(workdir, exist_ok=True)
os.chdir(workdir)
# 生成 3x1 网格路网
os.system('netgenerate --grid --grid.x-number=3 --grid.y-number=1 '
          '--grid.length=1000 --output-file=grid.net.xml 2>&1 | tail -1')
# 路由文件：4 车编队
with open('routes.rou.xml', 'w') as f:
    f.write('''<routes>
    <vType id="car" accel="2.0" decel="4.5" maxSpeed="30" length="4.5"/>
    <route id="r0" edges="A0B0 B0C0"/>
    <vehicle id="veh0" type="car" route="r0" depart="0" departPos="1500" departSpeed="15"/>
    <vehicle id="veh1" type="car" route="r0" depart="0" departPos="1483" departSpeed="15"/>
    <vehicle id="veh2" type="car" route="r0" depart="0" departPos="1466" departSpeed="15"/>
    <vehicle id="veh3" type="car" route="r0" depart="0" departPos="1449" departSpeed="15"/>
</routes>''')
with open('sumo.sumocfg', 'w') as f:
    f.write('''<configuration>
    <input><net-file value="grid.net.xml"/><route-files value="routes.rou.xml"/></input>
    <time><begin value="0"/><step-length value="0.05"/></time>
</configuration>''')
print('✓ 路网与路由就绪')


## 3. 真 TraCI 闭环仿真 | Real TraCI Closed Loop
启动 SUMO，TraCI 每 0.05s：
1. 读取 4 车位置/速度（真实 SUMO 物理）
2. 头车在 5-10s 减速（制造扰动）
3. 跟随车用你的 CACC 算法计算加速度、设置速度
观察扰动是否被放大——这就是弦稳定性测试。


In [ ]:
import traci
traci.start(['sumo', '-c', 'sumo.sumocfg', '--no-warnings', '--no-step-log'])
controllers = [CACCController(config={'headway': 0.8}) for _ in range(3)]
max_spacing_error = 0.0
history = []
vids = ['veh0', 'veh1', 'veh2', 'veh3']

for step in range(400):  # 20s
    traci.simulationStep()
    try:
        pos = {v: traci.vehicle.getPosition(v)[0] for v in vids}
        spd = {v: traci.vehicle.getSpeed(v) for v in vids}
    except:
        continue
    # Leader 扰动：5-10s 减速
    t = step * 0.05
    if 5 < t < 10:
        traci.vehicle.setSpeed('veh0', max(8, spd['veh0'] - 0.5 * 0.05))
    # Followers CACC
    for i in range(1, 4):
        ego, lead = vids[i], vids[i-1]
        obs = {'ego': {'x': pos[ego], 'v': spd[ego]}, 'targets': [{'x': pos[lead], 'v': spd[lead]}]}
        act = controllers[i-1].step(obs)
        v_new = max(0, spd[ego] + act['target_a'] * 0.05)
        traci.vehicle.setSpeed(ego, v_new)
        traci.vehicle.setLaneChangeMode(ego, 0)
        gap = pos[lead] - pos[ego] - 4.5
        max_spacing_error = max(max_spacing_error, abs(gap - (spd[ego]*0.8 + 5.0)))
    history.append((t, [spd[v] for v in vids]))

traci.close()
print(f'✓ TraCI 闭环完成：{len(history)} 步')
print(f'  最大间距误差 = {max_spacing_error:.2f}m')


## 4. 评测断言

In [ ]:
assert max_spacing_error <= 1.0, f'误差 {max_spacing_error:.2f}m 超标!'
print('✅ 评测通过！真 SUMO 编队稳定')


## 5. 可视化

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
times = [t for t, vs in history]
speeds = np.array([vs for t, vs in history])
plt.figure(figsize=(12, 5))
for i, name in enumerate(['Leader', 'Follower 1', 'Follower 2', 'Follower 3']):
    plt.plot(times, speeds[:, i], label=name)
plt.xlabel('Time (s)')
plt.ylabel('Speed (m/s)')
plt.title('SUMO Platoon - Real TraCI Closed Loop')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 🏆 挑战任务
1. 改 headway 为 0.6/1.0，对比弦稳定性。
2. 让 Leader 做正弦扰动，看编队响应。
